# Project 73 — Prompt vs Fine-Tune Comparison Lab
## Zero-Shot → Few-Shot → Detailed Instruction → Benchmark

**Stack:** LangChain · Ollama · pandas · Jupyter

In [ ]:
# !pip install -q langchain langchain-ollama pandas

## Step 1 — Define Evaluation Tasks

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import pandas as pd, time

llm = ChatOllama(model="qwen3:8b", temperature=0.1)

# Ground-truth labeled test set
test_data = [
    ("I absolutely love this product — best purchase ever!", "positive"),
    ("Terrible experience. Waste of money, would not recommend.", "negative"),
    ("It's okay, does what it's supposed to do, nothing special.", "neutral"),
    ("Exceeded my expectations! Five stars!", "positive"),
    ("Broke after two days. Horrible quality.", "negative"),
    ("Decent value for the price. Not amazing, not bad.", "neutral"),
    ("Amazing customer service and fast shipping!", "positive"),
    ("Worst product I've ever used. Total garbage.", "negative"),
    ("Works fine. Meets basic requirements.", "neutral"),
    ("This changed my life — I can't go back!", "positive"),
    ("Don't waste your money. Completely useless.", "negative"),
    ("Average product. You get what you pay for.", "neutral"),
]
print(f"Test set: {len(test_data)} labeled samples")
print(f"Distribution: {pd.Series([l for _,l in test_data]).value_counts().to_dict()}")

## Step 2 — Define Four Prompt Strategies

In [ ]:
strategies = {
    "zero_shot": ChatPromptTemplate.from_template(
        "Classify the sentiment as positive, negative, or neutral: {text}\nLabel:"
    ),
    "few_shot": ChatPromptTemplate.from_template(
        """Classify sentiment. Examples:
"Great product!" → positive
"Horrible quality" → negative
"It's fine" → neutral
"Love it!" → positive
"Terrible" → negative

Text: {text}
Label:"""
    ),
    "cot": ChatPromptTemplate.from_template(
        """Analyze the sentiment step by step:
1. Identify emotional words
2. Determine overall tone
3. Classify as positive/negative/neutral

Text: {text}
Analysis and Label:"""
    ),
    "detailed_instruction": ChatPromptTemplate.from_template(
        """You are a sentiment classification model trained on product reviews.
Classification rules:
- "positive" = satisfaction, praise, excitement, recommendation, happiness
- "negative" = dissatisfaction, complaint, frustration, warning against buying
- "neutral" = factual, indifferent, balanced pros/cons, acceptable

CRITICAL: Respond with ONLY ONE WORD: positive, negative, or neutral.

Text: {text}
Label:"""
    ),
}
print(f"Strategies: {list(strategies.keys())}")

## Step 3 — Run All Strategies

In [ ]:
results = []
for strat_name, prompt in strategies.items():
    chain = prompt | llm | StrOutputParser()
    correct = 0
    start = time.time()
    for text, expected in test_data:
        predicted = chain.invoke({"text": text}).strip().lower()
        # Extract label from response
        for label in ["positive", "negative", "neutral"]:
            if label in predicted:
                predicted = label
                break
        match = predicted == expected
        correct += match
        results.append({
            "strategy": strat_name, "text": text[:40],
            "expected": expected, "predicted": predicted,
            "correct": match,
        })
    elapsed = time.time() - start
    acc = correct / len(test_data)
    print(f"  {strat_name:<25} accuracy={acc:.0%} ({correct}/{len(test_data)}) in {elapsed:.1f}s")

## Step 4 — Detailed Analysis

In [ ]:
df = pd.DataFrame(results)

# Per-strategy metrics
print("STRATEGY COMPARISON")
print("=" * 60)
for strat in strategies:
    sub = df[df["strategy"] == strat]
    acc = sub["correct"].mean()
    errors = sub[~sub["correct"]]
    print(f"\n{strat}: {acc:.0%} accuracy")
    if len(errors) > 0:
        print(f"  Errors:")
        for _, e in errors.iterrows():
            print(f"    '{e['text']}' → expected={e['expected']}, got={e['predicted']}")

# Per-label accuracy
print("\nPER-LABEL ACCURACY:")
for label in ["positive", "negative", "neutral"]:
    sub = df[df["expected"] == label]
    by_strat = sub.groupby("strategy")["correct"].mean().round(2)
    print(f"  {label}: {by_strat.to_dict()}")

## Step 5 — Winner & Recommendations

In [ ]:
summary = df.groupby("strategy")["correct"].mean().sort_values(ascending=False)
print("LEADERBOARD")
print("=" * 40)
for i, (strat, acc) in enumerate(summary.items(), 1):
    medal = ["🥇","🥈","🥉","  "][min(i-1,3)]
    print(f"  {medal} {strat:<25} {acc:.0%}")

winner = summary.index[0]
print(f"\nBest strategy: {winner}")
print(f"\nRecommendation: {'Fine-tuning likely unnecessary' if summary.iloc[0] > 0.85 else 'Consider fine-tuning'}")

## What You Learned
- **Four prompt strategies** compared on same data
- **Per-label accuracy** to spot systematic weaknesses
- **Fine-tuning necessity assessment** based on prompt ceiling
- **Error analysis** for targeted improvement